# **Correction · TP7 | Signature RSA et OpenSSL**

Ce TP va utiliser OpenSSL.
Sous Windows, il faut probablement [le télécharger et l'installer](https://www.ssldragon.com/how-to/openssl/install-opnessl-windows/). On va :

- générer une clé
- faire le lien avec le cours
- signer/vérifier un fichier

*Corrigé rédigé en l'absence de corrigé officiel. Les réponses ont été vérifiées en Python ; les cellules qui lisent un fichier compagnon (`clear.txt`, `longclear.txt`, le module compagnon pour `hexdump`) n'ont pas pu être exécutées ici et sont à lancer dans le dossier du TP.*

*Choix pour la partie 7.3 : on signe avec la clé de 512 bits, pour pouvoir observer la limite de taille en 7.3.c.*

## 7.1. Clé RSA avec OpenSSL

### 7.1.a. Génération de clé privée

Pour générer une clé RSA, on peut utiliser la commande [genrsa de OpenSSL](https://docs.openssl.org/3.0/man1/openssl-genrsa/).

**TODO**  
Utiliser OpenSSL pour générer une clé RSA de 2048 bits dans `private_key_2048.pem` (112 bits de sécurité) ou une de 512 bits dans `private_key_512.pem` (64 bits de sécurité).

On peut utiliser la syntaxe `!<commande bash>` pour utiliser OpenSSL depuis ce notebook.

*Remarque :* Avec OpenSSL, la génération aléatoire n'est pas contrôlée, on ne peut pas fixer la *seed*. Pour que tout le monde dans le groupe ait la même clé, il faudrait utiliser un autre outil, [comme GnuTLS](https://stackoverflow.com/a/59329138).

In [ ]:
!openssl genrsa -out private_key_2048.pem 2048
!openssl genrsa -out private_key_512.pem 512

### 7.1.b. Génération de clé publique

**TODO**\
Avec `openssl pkey -pubout`, générer la clé publique correspondante dans `public_key.pem`.

In [ ]:
!openssl pkey -in private_key_2048.pem -pubout -out public_key.pem
!openssl pkey -in private_key_512.pem -pubout -out public_key_512.pem

### 7.1.c. Affichage de la clé publique

**TODO**\
Grâce à l'option `-text` de la commande `openssl pkey`, afficher la clé publique dans un format lisible. Vérifier que cette clé n'inclut pas l'exposant privé `d`.

In [ ]:
!openssl pkey -pubin -in public_key.pem -text -noout

> La clé publique ne contient que le **modulus** `n` et l'**exposant public** `e` (65537). **Pas de `d`.**

### 7.1.d. Affichage de la clé privée

**TODO**\
Grâce à l'option `-text` de la commande `openssl pkey`, afficher la clé privée dans un format lisible. Est-ce que cette clé ne contient que les informations nécessaires à la signature ?

In [ ]:
!openssl pkey -in private_key_2048.pem -text -noout

> Non, elle contient plus que `(n, d)`, seule paire nécessaire pour signer :
> `modulus` (n), `publicExponent` (e), `privateExponent` (d), `prime1` (p), `prime2` (q),
> `exponent1` (dP), `exponent2` (dQ) et `coefficient` (qInv).
> Les cinq derniers servent à **accélérer** la signature par les restes chinois (partie A.2).
> Ils sont aussi secrets que `d` : avec `p` et `q`, on recalcule `d`.

## 7.2. Analyser la clé

### 7.2.a. Extraction d'informations

**TODO**\
En vous inspirant du TP2 pour les paramètres Diffie-Hellman, extraire les informations de la clé privée et de la clé publique :
- le modulus `n`
- l'exposant public `e`
- l'exposant privé `d`
- les facteurs premiers `(p, q)` de `n`

In [ ]:
from cryptography.hazmat.primitives.serialization import load_pem_private_key

with open("private_key_2048.pem", "rb") as f:
    rsa_key_data = f.read()

rsa_key = load_pem_private_key(rsa_key_data, None)
rsa_pnums = rsa_key.private_numbers()
rsa_pubnums = rsa_key.public_key().public_numbers()

n, e = rsa_pubnums.n, rsa_pubnums.e          # clé publique
d, p, q = rsa_pnums.d, rsa_pnums.p, rsa_pnums.q   # secrets
print("n :", n.bit_length(), "bits · e =", e)
print("p :", p.bit_length(), "bits · q :", q.bit_length(), "bits")

### 7.2.b. Vérifier le modulus

**TODO**  
Vérifier qu'on a bien $n = p \times q$ avec $p$ et $q$ premiers (on pourra utiliser [SymPy](https://docs.sympy.org/latest/modules/ntheory.html#sympy.ntheory.primetest.isprime) ou [`galois`](https://mhostetter.github.io/galois/v0.4.6/api/galois.is_prime/#)).

In [ ]:
from sympy import isprime

print("n = p × q :", n == p * q)
print("p premier :", isprime(p), "· q premier :", isprime(q))

### 7.2.c. Exposant public, exposant privé

**TODO**  
À partir de ce qu'on a vu en cours, retrouver la valeur de l'exposant privé $d$ à partir de $p$, $q$ et $e$.

In [ ]:
from math import lcm

phi = (p - 1) * (q - 1)
d_cours = pow(e, -1, phi)                 # le cours : d = inv_φ(n)(e)
print("d du cours == d d'OpenSSL :", d_cours == d)

# Certaines versions d'OpenSSL calculent d modulo λ(n) = ppcm(p−1, q−1), un diviseur de φ(n).
# Les deux exposants signent de la même façon ; celui d'OpenSSL est alors simplement plus petit.
lam = lcm(p - 1, q - 1)
print("d ≡ d_cours (mod λ) :", d % lam == d_cours % lam)

## 7.3. Utiliser la clé

### 7.3.a. Signature et vérification

**TODO**  
Avec la commande [`openssl pkeyutl`](https://docs.openssl.org/3.0/man1/openssl-pkeyutl/#sm2), générer une signature pour le fichier `clear.txt`, à sauvegarder dans le fichier `sig.enc`. Vérifier la signature avec la clé publique (option `-pubin`).

In [ ]:
# pkeyutl signe directement le contenu du fichier, avec le padding PKCS#1 v1.5
!openssl pkeyutl -sign -inkey private_key_512.pem -in clear.txt -out sig.enc
!openssl pkeyutl -verify -pubin -inkey public_key_512.pem -in clear.txt -sigfile sig.enc

> La vérification affiche `Signature Verified Successfully`. Si on modifie `clear.txt`, elle échoue.

### 7.3.b. Padding de signature

On peut récupérer le fichier «de vérification» avec l'option `-verifyrecover`. Mais on peut aussi utiliser ce qu'on a appris en cours !

**TODO**  
En utilisant la fonction `pow` modulaire, effectuer cette vérification « à la main ». On pourra afficher le résultat avec la fonction `hexdump` du module compagnon.

*Remarque :* Le cryptosystème RSA peut aussi service au chiffrement de données. Dans ce cas, le padding devient aléatoire. On peut le vérifier avec la commande `openssl pkeyutl -encrypt` puis en déchiffrant les données $c$ à la main en calculant $c^e \% n$.

In [ ]:
with open("private_key_512.pem", "rb") as f:
    k512 = load_pem_private_key(f.read(), None)
n5, e5 = k512.public_key().public_numbers().n, k512.public_key().public_numbers().e

with open("sig.enc", "rb") as f:
    s = int.from_bytes(f.read(), "big")      # la signature est un grand entier

v = pow(s, e5, n5)                            # vérification du cours : v = s^e % n
v_bytes = v.to_bytes((n5.bit_length() + 7) // 8, "big")
print(v_bytes.hex(" "))                       # 00 01 ff ff … ff 00 puis le contenu de clear.txt
print(v_bytes[v_bytes.index(b"\x00", 2) + 1:])   # ce qui suit le séparateur 00 : le message

> On retrouve le **padding PKCS#1 v1.5 de signature** : `00 01`, une suite d'octets `ff`, un séparateur `00`,
> puis le contenu de `clear.txt`. Le calcul du cours `s^e % n` suffit à vérifier la signature.

### 7.3.c. Signature de hachage

**TODO**\
Que se passe-t-il si on essaie de signer des données qui font plus de 512 bits ? (On peut essayer avec `longclear.txt`)

In [ ]:
!openssl pkeyutl -sign -inkey private_key_512.pem -in longclear.txt -out sig_long.enc

# La solution : signer un haché de taille fixe (32 octets pour SHA-256)
!openssl dgst -sha256 -sign private_key_512.pem -out sig_long.enc longclear.txt
!openssl dgst -sha256 -verify public_key_512.pem -signature sig_long.enc longclear.txt

> Le message est trop long : RSA ne peut signer qu'un nombre **inférieur à n** (512 bits ici, moins 11 octets de padding).
> OpenSSL refuse (selon la version, *data too large for key size* ou *input data looks too long to be a hash*).
> En pratique, **on signe le haché** du message, de taille fixe, avec `openssl dgst -sha256 -sign`.

## Pour aller plus loin

Faisons le lien entre les formes de clé privée `(n, d)` et `(p, q, dP, dQ, qInv)`.

### A.1. `qInv`

Avec la version de la clé privée `(p, q, dP, dQ, qInv)`, on devrait avoir
$$ \mathtt{dP} = d \% \varphi(p), \qquad \mathtt{dQ} = d \% \varphi(q), \qquad \mathtt{qInv} = \mathrm{inv}_p(q). $$

**TODO**  
Vérifier avec les attributs `dmp1`, `dmp1` et `iqmp` que ces identités sont bien vérifiées dans votre clé générée.

In [ ]:
print("dP   = d % (p−1)   :", rsa_pnums.dmp1 == d % (p - 1))   # φ(p) = p − 1 car p est premier
print("dQ   = d % (q−1)   :", rsa_pnums.dmq1 == d % (q - 1))
print("qInv = inv_p(q)    :", rsa_pnums.iqmp == pow(q, -1, p))

### A.2. Signature

Les opérations décrites par la primitive de signature [RSASP1](https://www.rfc-editor.org/rfc/rfc8017#section-5.2.1) sont

```txt
    1.  Let s_p = m^dP mod p and s_q = m^dQ mod q.

    2.  Let h = (s_p - s_q) * qInv mod p.

    3.  Let s = s_q + q * h.
```

**TODO**  
Vérifier qu'on retrouve la même signature qu'avec OpenSSL.

Cette approche est une conséquence du [théorème de Sunzi](https://fr.wikipedia.org/wiki/Th%C3%A9or%C3%A8me_des_restes_chinois) (ou théorème des restes chinois) qui revient à dire que l'anneau résiduel $\mathbb{Z}_{pq}$ se comporte comme les couples d'entiers résiduels dans $\mathbb{Z}_p \times \mathbb{Z}_q$.

In [ ]:
pn5 = k512.private_numbers()
p5, q5 = pn5.p, pn5.q
m = v                                          # le message paddé retrouvé en 7.3.b

s_p = pow(m, pn5.dmp1, p5)                     # 1. deux petites exponentiations
s_q = pow(m, pn5.dmq1, q5)
h = (s_p - s_q) * pn5.iqmp % p5                # 2. recombinaison (restes chinois)
s_crt = s_q + q5 * h                           # 3.

print("même signature qu'OpenSSL :", s_crt == s)
print("même résultat que m^d % n :", s_crt == pow(m, pn5.d, n5))

### A.3. Fichier PEM en standard ASN.1

La clé est écrite en Base64 dans des fichiers [PEM](https://en.wikipedia.org/wiki/Privacy-Enhanced_Mail). Même une fois convertie en liste d'octets elle est peu compréhensible, puisqu'il s'agit d'un format de données assez brut : [ASN.1](https://letsencrypt.org/docs/a-warm-welcome-to-asn1-and-der/). Heureusement, certains outils permettent de l'analyser, comme [celui de Lapo Luchini](https://lapo.it/asn1js/).

**TODO**  
À l'aide de :

- la [référence](https://www.rfc-editor.org/rfc/rfc8017#appendix-A.1.2) de clé privée RSA
- un outil d'analyse de [ASN.1](https://lapo.it/asn1js/)

déterminer les valeurs des différents nombres utilisés avec RSA.

Vérifier que cela correspond à `openssl pkey -text -in private_key_2048.pem` (ou `private_key_512.pem`).

In [ ]:
# Le PEM d'OpenSSL 3 est au format PKCS#8 : une enveloppe qui contient la clé RSA (PKCS#1)
!openssl asn1parse -in private_key_512.pem
# On ouvre l'OCTET STRING (à l'offset 22) pour voir les 9 entiers de RSAPrivateKey
!openssl asn1parse -in private_key_512.pem -strparse 22

> Dans l'ordre de la RFC 8017 (A.1.2) : `version` (0), `modulus` n, `publicExponent` e, `privateExponent` d,
> `prime1` p, `prime2` q, `exponent1` dP, `exponent2` dQ, `coefficient` qInv.
> Ce sont exactement les valeurs affichées par `openssl pkey -text`, en hexadécimal.